# QuestionGen Pending Runner

Launcher-only notebook for Google Colab and Google Drive.

This pending runner is the subtype-expanded release notebook. Broad family selections still come from `QUESTION_TYPES`, but each family now expands into one or more concrete subtype rows during batch execution.

Use this notebook to run the full batch path safely from Drive, and to launch the Gradio app later without moving Drive or secret handling into `src/questiongen/`.


## 1. Mount Drive And Define Paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

# Edit DATA_DIR if your runtime folder lives somewhere else in Drive.
DATA_DIR = Path("/content/drive/MyDrive/QuestionGenData")
SECRETS_DIR = DATA_DIR / "secrets"
INPUT_DIR = DATA_DIR / "input"
OUTPUT_DIR = DATA_DIR / "output"
LOGS_DIR = DATA_DIR / "logs"

for folder in [SECRETS_DIR, INPUT_DIR, OUTPUT_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

API_KEY_FILE = SECRETS_DIR / "api_key.txt"
INPUT_CSV = INPUT_DIR / "questions.csv"
OUTPUT_CSV = OUTPUT_DIR / "generated_questions.csv"
OUTPUT_JSON = OUTPUT_DIR / "generated_questions.json"
OUTPUT_MD = OUTPUT_DIR / "generated_questions.md"

print("DATA_DIR:", DATA_DIR)
print("API_KEY_FILE:", API_KEY_FILE)
print("INPUT_CSV:", INPUT_CSV)
print("OUTPUT_CSV:", OUTPUT_CSV)
print("OUTPUT_JSON:", OUTPUT_JSON)
print("OUTPUT_MD:", OUTPUT_MD)


## 2. Load Secrets From Drive

In [ ]:
import os

def load_api_keys(filepath: str | Path) -> None:
    filepath = Path(filepath)

    if not filepath.exists():
        raise FileNotFoundError(
            f"API key file not found: {filepath}\n"
            "Create it with lines like OPENAI_API_KEY=sk-..."
        )

    with filepath.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            if "=" not in line:
                continue

            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()

load_api_keys(API_KEY_FILE)

MODEL_NAME = os.getenv("QUESTIONGEN_MODEL", "gpt-5-mini")
TEMPERATURE = float(os.getenv("QUESTIONGEN_TEMPERATURE", "0"))

print("Loaded env vars:")
print("- OPENAI_API_KEY:", "set" if os.getenv("OPENAI_API_KEY") else "missing")
print("- QUESTIONGEN_MODEL:", MODEL_NAME)
print("- QUESTIONGEN_TEMPERATURE:", TEMPERATURE)


## 3. Clone And Install The Repo

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/AwkAwkAardvark/QuestionGen.git"
REPO_DIR = Path("/content/QuestionGen")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
get_ipython().run_line_magic("pip", f"install -e {REPO_DIR}")
get_ipython().run_line_magic("pip", "install 'gradio>=5.0.0'")

# Colab sometimes installs an editable package successfully but does not
# expose the new src path to the running kernel immediately.
SRC_DIR = REPO_DIR / "src"
importlib.invalidate_caches()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if importlib.util.find_spec("questiongen") is None:
    raise ModuleNotFoundError(
        "questiongen is still not importable after installation. "
        "Check the install cell output before continuing."
    )

print("Cloned and installed:", REPO_DIR)
print("Source path added:", SRC_DIR)
print("Python executable:", sys.executable)


## 4. Run The Current Batch Pipeline

In [ ]:
import json
from collections import Counter

from questiongen.batch import run_batch_files
from questiongen.config import create_structured_llm
from questiongen.graph import compile_question_graph
from questiongen.question_types import QUESTION_TYPES

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")

question_type_keys = list(QUESTION_TYPES.keys())

# Broad family keys remain the launcher surface. The batch layer now expands
# them into concrete subtype rows automatically.

runner = compile_question_graph(
    structured_llm_factory=lambda schema: create_structured_llm(
        schema,
        model_name=MODEL_NAME,
        temperature=TEMPERATURE,
    )
)

results = run_batch_files(
    input_csv=str(INPUT_CSV),
    output_csv=str(OUTPUT_CSV),
    question_type_keys=question_type_keys,
    runner=runner,
    output_markdown=str(OUTPUT_MD),
)

OUTPUT_JSON.write_text(
    json.dumps([result.model_dump() for result in results], ensure_ascii=False, indent=2),
    encoding="utf-8",
)

status_counts = Counter(result.status for result in results)
print("Question families:", question_type_keys)
print("Expanded subtype rows:", len(results))
print("Status counts:", dict(status_counts))
print("CSV:", OUTPUT_CSV)
print("JSON:", OUTPUT_JSON)
print("Markdown:", OUTPUT_MD)


## 5. Preview Outputs

The CSV/JSON previews now include `QuestionFormatKey`, `QuestionSubtypeKey`, and `QuestionSubtype`, so one broad family may appear multiple times per source row.


In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)
df.head()


In [ ]:
print(OUTPUT_MD.read_text(encoding="utf-8")[:3000])
print(OUTPUT_JSON.read_text(encoding="utf-8")[:3000])


## 6. Pending Gradio Hook

In [ ]:
from questiongen.ui.gradio_app import create_app

try:
    app = create_app()
except NotImplementedError as exc:
    print("Gradio app is still pending:")
    print(exc)
else:
    app.launch(debug=True)


## 7. Download Outputs

In [ ]:
from google.colab import files

files.download(str(OUTPUT_CSV))
files.download(str(OUTPUT_JSON))
files.download(str(OUTPUT_MD))
